# Qualcomm® AI Hub: NPU-Optimized Video Model Pipeline

**Goal:** Compile, quantize (INT8), profile, and deploy the R(2+1)D-18 model
to achieve **<34ms** inference on the Dragonwing IQ-9075 EVK (QCS9100).

**Key optimizations baked into the ONNX model:**
- Input layout: `(1, 16, 112, 112, 3)` — NDHWC (Hexagon NPU native)
- Normalization baked into graph (zero CPU overhead)
- INT8 quantization via calibration data

**Files needed from the Training notebook:**
1. `qualcomm_r2plus1d.onnx` — the NDHWC wrapper model
2. `calibration_data/calibration_inputs.npy` — 100 samples for INT8 quantization

In [1]:
import sys
!{sys.executable} -m pip install -q qai-hub onnx

In [2]:
!qai-hub configure --api_token wyfb5d78py0bgow6gz10krdqyd8qq0l9wlfw9swc

qai-hub configuration saved to C:\Users\USER/.qai_hub/client.ini
==================== C:\Users\USER/.qai_hub/client.ini ====================
[api]
api_token = wyfb5d78py0bgow6gz10krdqyd8qq0l9wlfw9swc
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli




2026-04-10 16:00:02.278 - INFO - Enabling verbose logging.
C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\qai_hub\_cli.py:412: UserWarning: Overwriting configuration: C:\Users\USER/.qai_hub/client.ini (previous configuration saved to C:\Users\USER/.qai_hub/client.ini.bak)
  warnings.warn(


In [3]:
import qai_hub as hub
import numpy as np
import os
print("AI Hub ready 🚀")

AI Hub ready 🚀


In [9]:
# =========================================================
# CONFIGURATION
# =========================================================
ONNX_PATH  = "qualcomm_r2plus1d.onnx"       # From training notebook
CALIB_PATH = "calibration_inputs.npy"  # 100 samples for INT8

# Target device — the LPCVC competition board
device = hub.Device("Dragonwing IQ-9075 EVK")

# Input spec — matching the ONNX model
INPUT_SHAPE = (1, 3, 16, 112, 112)

print(f'📱 Device: {device}')
print(f'📐 Input shape: {INPUT_SHAPE}')
print(f'📄 ONNX model: {ONNX_PATH}')
print(f'📊 Calibration data: {CALIB_PATH}')

📱 Device: Device(name='Dragonwing IQ-9075 EVK', os='', attributes=[])
📐 Input shape: (1, 3, 16, 112, 112)
📄 ONNX model: qualcomm_r2plus1d.onnx
📊 Calibration data: calibration_inputs.npy


---
## Step 1: FP16 Compile + Profile (Baseline)
First, let's establish the FP16 baseline to confirm the layout optimization works.

In [10]:
# =========================================================
# FP16 COMPILE (baseline — should already be much faster)
# =========================================================
print('🔧 Compiling FP16 model for NPU...')

compile_job_fp16 = hub.submit_compile_job(
    model=ONNX_PATH,
    device=device,
    input_specs={
        "input": INPUT_SHAPE   # (1, 3, 16, 112, 112) NCDHW
    },
    options="--target_runtime tflite"
)

target_model_fp16 = compile_job_fp16.get_target_model()
print(f'✅ FP16 compilation complete!')

🔧 Compiling FP16 model for NPU...
Uploading qualcomm_r2plus1d.onnx


100%|██████████| 120M/120M [01:44<00:00, 1.20MB/s] 


Scheduled compile job (jperyddog) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jperyddog/

Waiting for compile job (jperyddog) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
✅ FP16 compilation complete!


In [8]:
# =========================================================
# FP16 PROFILE — check if layout fix alone hits <34ms
# =========================================================
print('📊 Profiling FP16 model on device...')

# Re-resolve the model in case the compile job failed earlier or the notebook
# has a newer compiled model available below.
target_model_fp16 = compile_job_fp16.get_target_model() if 'compile_job_fp16' in dir() else None
target_model_int8 = locals().get('target_model_int8', None)

best_model = target_model_fp16 if target_model_fp16 is not None else target_model_int8

if best_model is None:
    print('⚠️ No compiled model is available yet. Run a successful compile cell before profiling.')
else:
    profile_job_fp16 = hub.submit_profile_job(
        model=best_model,
        device=device
    )

    print(f'📊 FP16 Profile job submitted!')
    print(f'   View results: https://workbench.aihub.qualcomm.com/jobs/{profile_job_fp16.job_id}/')

📊 Profiling FP16 model on device...
⚠️ No compiled model is available yet. Run a successful compile cell before profiling.


---
## Step 2: INT8 Quantized Compile + Profile (Target <34ms)
If FP16 is still above 34ms, INT8 quantization gives another 2-3x speedup.

In [ ]:
# =========================================================
# FIXED INT8 CALIBRATION DATA
# =========================================================
print('📊 Loading calibration data...')
calib_data = np.load(CALIB_PATH)  # Expected: (100, 3, 16, 112, 112)

# The Hub expects a dict: { "input_name": [sample1, sample2, ... sampleN] }
# Each sample inside the list must match the expected input shape (1, 3, 16, 112, 112)
calibration_dataset = {
    "input": [calib_data[i:i+1] for i in range(len(calib_data))]
}

print(f'✅ Prepared {len(calibration_dataset["input"])} calibration samples in the correct format.')

# Now the compile job will work
compile_job_int8 = hub.submit_compile_job(
    model=ONNX_PATH,
    device=device,
    input_specs={"input": INPUT_SHAPE},
    options="--target_runtime tflite --quantize_full_type int8 --quantize_io",
    calibration_data=calibration_dataset  # Now passed as a dict
)

📊 Loading calibration data...
✅ Prepared 100 calibration samples in the correct format.
Uploading qualcomm_r2plus1d.onnx


 94%|█████████▎| 112M/120M [02:05<00:07, 1.14MB/s] 

100%|██████████| 120M/120M [01:10<00:00, 1.77MB/s]
Uploading dataset: 156MB [01:35, 1.71MB/s]                          


Scheduled compile job (j57j7wjn5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j57j7wjn5/



In [10]:
# =========================================================
# INT8 PROFILE — Fixed with Wait Logic
# =========================================================

# 1. Ensure the compile job finished and get the model
print("⏳ Waiting for compilation to finish...")
# This line ensures target_model_int8 is defined from the previous job
target_model_int8 = compile_job_int8.get_target_model() 

print('📊 Profiling INT8 model on device...')

# 2. Submit the profile job
profile_job_int8 = hub.submit_profile_job(
    model=target_model_int8,
    device=device
)

print(f'🚀 INT8 Profile job submitted!')
print(f'   Job ID: {profile_job_int8.job_id}')
print(f'   View live progress: https://app.aihub.qualcomm.com/jobs/{profile_job_int8.job_id}/')

# 3. Optional: Block and wait for results
print("\n🕒 Waiting for latency results (usually takes 2-5 mins)...")
profile_results = profile_job_int8.download_profile()
latency_ms = profile_results["compute_metrics"]["cpu_and_npu_latency"]["mean"] / 1000

print(f"\n{'='*30}")
print(f"🔥 INT8 LATENCY: {latency_ms:.2f} ms")
print(f"{'='*30}")

⏳ Waiting for compilation to finish...
Waiting for compile job (j57j7wjn5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
📊 Profiling INT8 model on device...
Scheduled profile job (jp4x9ox25) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp4x9ox25/

🚀 INT8 Profile job submitted!
   Job ID: jp4x9ox25
   View live progress: https://app.aihub.qualcomm.com/jobs/jp4x9ox25/

🕒 Waiting for latency results (usually takes 2-5 mins)...
Waiting for profile job (jp4x9ox25) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


KeyError: 'compute_metrics'

---
## Step 3: Inference Verification
Run a dummy input through the deployed model to verify correctness.

In [11]:
# =========================================================
# ON-DEVICE INFERENCE TEST
# =========================================================
# Use the INT8 model if available, otherwise FP16
best_model = target_model_int8 if 'target_model_int8' in dir() else target_model_fp16

# Create a test input matching the NDHWC shape
dummy_input = np.random.rand(*INPUT_SHAPE).astype("float32")

print('🔄 Running inference on device...')
inference_job = hub.submit_inference_job(
    model=best_model,
    device=device,
    inputs={"input": [dummy_input]}
)

output = inference_job.download_output_data()

# Show top-5 predictions
logits = list(output.values())[0][0]  # First output, first batch
top5_idx = np.argsort(logits[0])[::-1][:5]
print(f'\n✅ Inference complete!')
print(f'   Output shape: {logits.shape}')
print(f'   Top-5 class indices: {top5_idx}')
print(f'   Top-5 logits: {logits[0][top5_idx]}')

🔄 Running inference on device...


Uploading dataset: 2.20MB [00:04, 551kB/s]                             460kB/s] 


Scheduled inference job (jpy47w40p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpy47w40p/

Waiting for inference job (jpy47w40p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpbva4spra.h5: 100%|██████████| 13.9k/13.9k [00:00<00:00, 1.34MB/s]


✅ Inference complete!
   Output shape: (1, 91)
   Top-5 class indices: [11 26 12 10 38]
   Top-5 logits: [-2.132744  -2.285083  -2.3866422 -3.0975568 -3.1991162]


In [ ]:
# =========================================================
# SUMMARY
# =========================================================
print('=' * 60)
print('📊 LPCVC OPTIMIZATION REPORT')
print('=' * 60)
print(f'Target Device:    Dragonwing IQ-9075 EVK (QCS9100)')
print(f'Input Layout:     NCDHW (1, 3, 16, 112, 112)')
print(f'Normalization:    Baked into model graph')
print(f'\nJobs:')
if 'profile_job_fp16' in dir():
    print(f'  FP16 Profile:   https://workbench.aihub.qualcomm.com/jobs/{profile_job_fp16.job_id}/')
if 'profile_job_int8' in dir():
    print(f'  INT8 Profile:   https://workbench.aihub.qualcomm.com/jobs/{profile_job_int8.job_id}/')
print(f'\n⚡ Check the profile links above to verify latency is <34ms!')
print('=' * 60)

📊 LPCVC OPTIMIZATION REPORT
Target Device:    Dragonwing IQ-9075 EVK (QCS9100)
Input Layout:     NDHWC (1, 16, 112, 112, 3)
Normalization:    Baked into model graph

Jobs:
  FP16 Profile:   https://workbench.aihub.qualcomm.com/jobs/jp1dvldnp/
  INT8 Profile:   https://workbench.aihub.qualcomm.com/jobs/jp4x9ox25/

⚡ Check the profile links above to verify latency is <34ms!
